# Tiến hóa schema: bằng chứng cho `docs/tien_hoa_schema.md`

Notebook này là **phần chứng minh** của `docs/tien_hoa_schema.md` (quy ước CLAUDE.md §3: `.md` diễn giải, notebook chứng minh).
Các số mang dấu † trong tài liệu được tính lại ở đây bằng cell chạy lại được.

Bộ dữ liệu: `student_data.zip` do giảng viên giao, giải nén ở `data/student_data/` (gitignored, không commit).

| Phần | Nội dung | Ứng với tài liệu |
|---|---|---|
| 1 | Kiểm kê 15 file, file nào trùng byte với `data/` | §2 |
| 2 | `orders_enriched.csv` | §2, §7 |
| 3 | `shipments_realistic.csv` | §2, §7 |
| 4 | `eprom.json` | §2, §7 |
| 5 | `epd.json`: đánh lại khóa | §2, §4 dòng 7, §7 |
| 6 | `tf.json`: phát lại bản ghi cũ | §2, §4 dòng 8, §7 |
| 7 | Nếu nạp epd theo id thì COGS lệch bao nhiêu | §7 |
| 8 | **Bằng chứng "trước"**: hệ thống hiện tại xử lý từng file ra sao | §3.2 |
| 9 | Tóm tắt số | — |

Phần 8 chạy `snapshot.prepare` và `contract.check` thật, nhưng trên **thư mục tạm**: không đụng `data/` hay `warehouse/`.
Mọi phép nối trên file lớn dùng DuckDB, đọc toàn chuỗi (`all_varchar`), giống cách pipeline đọc snapshot.

In [1]:
import pandas as pd, numpy as np
import warnings; warnings.filterwarnings('ignore')
pd.set_option('display.width', 220); pd.set_option('display.max_columns', 60)

DATA = 'data'
R = lambda f, **k: pd.read_csv(f'{DATA}/{f}', low_memory=False, **k)

In [2]:
import sys, json, hashlib, re, shutil, tempfile
from pathlib import Path
import duckdb

SD = f'{DATA}/student_data'
con = duckdb.connect()
Q = lambda sql: con.sql(sql).df()
one = lambda sql: con.sql(sql).fetchone()[0]

for name in ['orders', 'shipments', 'geography', 'products', 'promotions', 'order_items', 'web_traffic', 'sales']:
    con.execute(f"create view {name} as select * from read_csv('{DATA}/{name}.csv', all_varchar=true)")
con.execute(f"create view oe as select * from read_csv('{SD}/orders_enriched.csv', all_varchar=true)")
con.execute(f"create view sr as select * from read_csv('{SD}/shipments_realistic.csv', all_varchar=true)")

def load_json(name):
    # Như adapter đề xuất ở §4 dòng 5: mỗi bản ghi thành một dòng, mọi giá trị thành chuỗi, None giữ là rỗng.
    rows = json.loads(Path(f'{SD}/{name}.json').read_text(encoding='utf-8'))
    df = pd.DataFrame([{k: (None if v is None else str(v)) for k, v in r.items()} for r in rows], dtype=object)
    df.insert(0, '_src_row', range(1, len(df) + 1))
    return df

eprom, epd, tf = load_json('eprom'), load_json('epd'), load_json('tf')
for name, df in [('eprom', eprom), ('epd', epd), ('tf', tf)]:
    con.register(name, df)
print(sys.executable, '| duckdb', duckdb.__version__)
print({n: len(d) for n, d in [('eprom', eprom), ('epd', epd), ('tf', tf)]})

D:\retail-analytics\.venv\Scripts\python.exe | duckdb 1.5.5
{'eprom': 1000, 'epd': 1000, 'tf': 1000}


## 1. Kiểm kê: 15 file, 10 file trùng byte với `data/`

So sha256 từng file trong zip với file cùng tên ở `data/`.

In [3]:
sha = lambda p: hashlib.sha256(Path(p).read_bytes()).hexdigest()
inv = []
for p in sorted(Path(SD).iterdir()):
    old = Path(DATA) / p.name
    inv.append({'file': p.name, 'bytes': p.stat().st_size, 'co_ten_trong_data': old.exists(),
                'trung_byte': old.exists() and sha(p) == sha(old)})
inv = pd.DataFrame(inv)
print('số file:', len(inv), '| trùng byte:', int(inv.trung_byte.sum()))
print('nguồn Datathon không có trong zip:', sorted({'sales.csv', 'sample_submission.csv', 'orders.csv', 'shipments.csv'} - set(inv.file)))
inv

số file: 15 | trùng byte: 10
nguồn Datathon không có trong zip: ['orders.csv', 'sales.csv', 'sample_submission.csv', 'shipments.csv']


,file,bytes,co_ten_trong_data,trung_byte
0,customers.csv,7079679,True,True
1,epd.json,224069,False,False
2,eprom.json,324919,False,False
3,geography.csv,1402281,True,True
4,inventory.csv,5668678,True,True
5,order_items.csv,23942735,True,True
6,orders_enriched.csv,155195353,False,False
7,payments.csv,18383288,True,True
8,products.csv,195223,True,True
9,promotions.csv,4444,True,True


## 2. `orders_enriched.csv`

Kiểm: BOM, 8 cột gốc có trùng `orders.csv` không, `city/region/district` có suy được từ `geography` qua `zip` không,
nhân viên có phải thực thể riêng không (id quyết định mọi thuộc tính), và `comment` có bao nhiêu giá trị khác nhau.

In [4]:
path = Path(f'{SD}/orders_enriched.csv')
first = path.open('rb').read(3)
with path.open(encoding='utf-8', newline='') as f: h_utf8 = f.readline().rstrip('\r\n').split(',')
with path.open(encoding='utf-8-sig', newline='') as f: h_sig = f.readline().rstrip('\r\n').split(',')
print('3 byte đầu:', first, '| là BOM UTF-8:', first == b'\xef\xbb\xbf')
print('cột đầu khi đọc utf-8    :', repr(h_utf8[0]))
print('cột đầu khi đọc utf-8-sig:', repr(h_sig[0]))
print('số cột:', len(h_sig), '| cột thêm so với orders:', [c for c in h_sig if c not in R('orders.csv', nrows=0).columns])

3 byte đầu: b'\xef\xbb\xbf' | là BOM UTF-8: True
cột đầu khi đọc utf-8    : '\ufefforder_id'
cột đầu khi đọc utf-8-sig: 'order_id'


số cột: 17 | cột thêm so với orders: ['city', 'region', 'district', 'sales_employee_id', 'sales_employee_name', 'marital_status', 'education_level', 'years_experience', 'comment']


In [5]:
BASE_O = ['order_date', 'customer_id', 'zip', 'order_status', 'payment_method', 'device_type', 'order_source']
cat = lambda a, cols: 'concat_ws(\'|\', ' + ', '.join(f'{a}.{c}' for c in cols) + ')'
Q(f'''
select count(*) as dong, count(distinct e.order_id) as order_id_khac_nhau, (select count(*) from orders) as dong_orders,
       count(*) filter (where o.order_id is null) as khong_co_trong_orders,
       count(*) filter (where o.order_id is not null and {cat('e', BASE_O)} <> {cat('o', BASE_O)}) as lech_8_cot_goc
from oe e left join orders o using (order_id)
''')

,dong,order_id_khac_nhau,dong_orders,khong_co_trong_orders,lech_8_cot_goc
0,646945,646945,646945,0,0


In [6]:
Q('''
select count(*) as dong,
       count(*) filter (where e.city = g.city and e.region = g.region and e.district = g.district) as khop_geography_qua_zip
from oe e left join geography g using (zip)
''')

,dong,khop_geography_qua_zip
0,646945,646945


In [7]:
def fd(view, key, attrs):
    # Số giá trị khóa có hơn một giá trị thuộc tính: 0 nghĩa là key → attr (phụ thuộc hàm đúng).
    return pd.Series({a: one(f'select count(*) from (select {key} from {view} group by 1 having count(distinct "{a}") > 1)')
                      for a in attrs}, name='vi_pham_FD')

EMP = ['sales_employee_name', 'marital_status', 'education_level', 'years_experience']
print('số nhân viên:', one('select count(distinct sales_employee_id) from oe'),
      '| dòng thiếu sales_employee_id:', one('select count(*) from oe where nullif(trim(sales_employee_id), \'\') is null'))
fd('oe', 'sales_employee_id', EMP)

số nhân viên: 200 | dòng thiếu sales_employee_id: 0


sales_employee_name    0
marital_status         0
education_level        0
years_experience       0
Name: vi_pham_FD, dtype: int64

In [8]:
print('comment khác nhau:', one('select count(distinct comment) from oe'),
      '| comment rỗng:', one('select count(*) from oe where nullif(trim(comment), \'\') is null'))
Q('select comment, count(*) as so_don from oe group by 1 order by 2 desc limit 5')

comment khác nhau: 18 | comment rỗng: 0


,comment,so_don
0,"Tư vấn chi tiết, thái độ chuyên nghiệp và thân...",48900
1,"Nhân viên giao tiếp tốt, xử lý yêu cầu đúng hẹ...",48779
2,"Chăm sóc khách hàng tốt, thời gian phản hồi ngắn.",48758
3,Khách hàng đánh giá cao sự chủ động và trách n...,48589
4,"Dịch vụ ổn định, nhân viên luôn sẵn sàng hỗ tr...",48486


## 3. `shipments_realistic.csv`

Kiểm: 4 cột gốc có trùng `shipments.csv` không, số shipper, phụ thuộc hàm của thuộc tính shipper, PII,
và `join_date` / `city` có phải ảnh chụp hiện tại của shipper chứ không phải thông tin tại lúc giao.

In [9]:
cols_sr = list(Q('select * from sr limit 0').columns)
SHIP_ATTR = [c for c in cols_sr if c not in ('shipper_id', 'order_id', 'ship_date', 'delivery_date', 'shipping_fee')]
print('số cột:', len(cols_sr), '| cột thêm:', len(cols_sr) - 4)
BASE_S = ['ship_date', 'delivery_date', 'shipping_fee']
Q(f'''
select count(*) as dong, (select count(*) from shipments) as dong_shipments,
       count(*) filter (where s.order_id is null) as khong_co_trong_shipments,
       count(*) filter (where s.order_id is not null and {cat('r', BASE_S)} <> {cat('s', BASE_S)}) as lech_4_cot_goc,
       count(distinct r.shipper_id) as so_shipper
from sr r left join shipments s using (order_id)
''')

số cột: 22 | cột thêm: 18


,dong,dong_shipments,khong_co_trong_shipments,lech_4_cot_goc,so_shipper
0,566067,566067,0,0,80


In [10]:
v = fd('sr', 'shipper_id', SHIP_ATTR)
print('thuộc tính shipper:', len(v), '| vi phạm FD:', int(v.sum()))
v.to_frame().T

thuộc tính shipper: 17 | vi phạm FD: 0


,shipper_company,shipper_vehicle,shipper_experience_years,shipper_rating,delivery_success_rate,average_delivery_time,working_shift,join_date,shipper_name,shipper_phone,shipper_gender,shipper_age,shipper_marital_status,shipper_education,city,region,district
vi_pham_FD,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0


In [11]:
print('cột nghi PII:', [c for c in cols_sr if c in ('shipper_name', 'shipper_phone', 'shipper_gender', 'shipper_age')])
Q('''
select min(cast(join_date as date)) as join_min, max(cast(join_date as date)) as join_max,
       count(*) filter (where cast(join_date as date) > cast(ship_date as date)) as join_sau_ship,
       count(*) as dong,
       round(100.0 * count(*) filter (where cast(join_date as date) > cast(ship_date as date)) / count(*), 1) as pct
from sr
''')

cột nghi PII: ['shipper_name', 'shipper_phone', 'shipper_gender', 'shipper_age']


,join_min,join_max,join_sau_ship,dong,pct
0,2018-01-12,2026-03-17,519276,566067,91.7


In [12]:
Q('''
select count(*) as dong,
       count(*) filter (where r.city <> g.city) as city_shipper_khac_city_don,
       round(100.0 * count(*) filter (where r.city <> g.city) / count(*), 1) as pct
from sr r join orders o using (order_id) join geography g on o.zip = g.zip
''')

,dong,city_shipper_khac_city_don,pct
0,566067,551578,97.4


## 4. `eprom.json`

Kiểm: cùng field với `promotions` không, id có va chạm hay được đơn nào tham chiếu không, và giá trị lạ.

In [13]:
prom = R('promotions.csv', dtype=str)
print('cùng tập field với promotions:', set(eprom.columns) - {'_src_row'} == set(prom.columns))
ids = eprom.promo_id.fillna('')
print('id có khoảng trắng hai đầu:', int((ids != ids.str.strip()).sum()), list(ids[ids != ids.str.strip()]))
print('id đúng khuôn PROMO-nnnn-nnnn (sau trim):', int(ids.str.strip().str.fullmatch(r'PROMO-\d{4}-\d{4}').sum()))
print('id trùng với promotions:', int(ids.str.strip().isin(prom.promo_id).sum()))
print('số promo gốc được nhân bản:', ids.str.strip().str[:10].nunique(), '| trung bình bản/promo:',
      round(len(eprom) / ids.str.strip().str[:10].nunique(), 1))
print('dòng order_items tham chiếu id eprom:', one('''
    select count(*) from order_items
    where trim(promo_id) in (select trim(promo_id) from eprom) or trim(promo_id_2) in (select trim(promo_id) from eprom)'''))

cùng tập field với promotions: True
id có khoảng trắng hai đầu: 1 ['   PROMO-0037-0712   ']
id đúng khuôn PROMO-nnnn-nnnn (sau trim): 1000
id trùng với promotions: 0
số promo gốc được nhân bản: 50 | trung bình bản/promo: 20.0


dòng order_items tham chiếu id eprom: 0


In [14]:
bad = eprom[pd.to_numeric(eprom.discount_value, errors='coerce').isna()]
print('discount_value không ép được sang số:', bad.discount_value.tolist())
print('promo_name rỗng:', eprom[eprom.promo_name.fillna('').str.strip().eq('')][['_src_row', 'promo_id', 'promo_name']].to_dict('records'))

discount_value không ép được sang số: ['twenty percent', '30%%']
promo_name rỗng: [{'_src_row': 18, 'promo_id': 'PROMO-0002-0018', 'promo_name': None}, {'_src_row': 287, 'promo_id': 'PROMO-0009-0287', 'promo_name': ''}]


## 5. `epd.json`: sản phẩm cũ mang id mới

Khớp từng bản ghi epd với `products` theo **đủ thuộc tính** (tên, category, segment, size, color, price, cogs), không theo id.
Nếu bản ghi khớp đúng một sản phẩm mà id lại khác, đó là **đánh lại khóa**.

In [15]:
con.execute('''
create or replace temp table epd_khop as
with e as (select _src_row, product_id, product_name, category, segment, size, color,
                  try_cast(price as double) as price, try_cast(cogs as double) as cogs from epd),
     p as (select product_id as pid, product_name, category, segment, size, color,
                  cast(price as double) as price, cast(cogs as double) as cogs from products)
select e._src_row, e.product_id as epd_id, count(p.pid) as n_khop, min(p.pid) as pid_khop
from e left join p on e.product_name = p.product_name and e.category = p.category and e.segment = p.segment
                  and e.size = p.size and e.color = p.color and abs(e.price - p.price) < 1e-6 and abs(e.cogs - p.cogs) < 1e-6
group by all
''')
print('cùng tập field với products:', set(epd.columns) - {'_src_row'} == set(R('products.csv', nrows=0).columns))
Q('select n_khop, count(*) as ban_ghi from epd_khop group by 1 order by 1')

cùng tập field với products: True


,n_khop,ban_ghi
0,0,5
1,1,995


In [16]:
Q('''
select count(*) filter (where n_khop = 1) as khop_dung_1,
       count(*) filter (where n_khop = 1 and epd_id = pid_khop) as khop_va_cung_id,
       count(*) filter (where try_cast(epd_id as integer) in (select cast(product_id as integer) from products)) as dong_co_id_trung_so_voi_products,
       count(distinct epd_id) filter (where try_cast(epd_id as integer) in (select cast(product_id as integer) from products)) as id_khac_nhau_trung_so,
       count(*) filter (where try_cast(epd_id as integer) in (select cast(product_id as integer) from products)
                        and (n_khop <> 1 or epd_id <> pid_khop)) as id_trung_so_nhung_khac_san_pham
from epd_khop
''')

,khop_dung_1,khop_va_cung_id,dong_co_id_trung_so_voi_products,id_khac_nhau_trung_so,id_trung_so_nhung_khac_san_pham
0,995,0,998,997,998


In [17]:
print('ví dụ: epd id 1 là sản phẩm', one("select pid_khop from epd_khop where epd_id = '1'"), 'trong products')
print('5 bản ghi không khớp sản phẩm nào (giá trị bị cấy):')
display(epd.merge(Q('select _src_row from epd_khop where n_khop = 0'), on='_src_row'))
print('product_id rỗng hoặc None:'); display(epd[epd.product_id.fillna('').str.strip().eq('')])
print('product_id trùng:'); display(epd[epd.product_id.notna() & epd.product_id.duplicated(keep=False)])

ví dụ: epd id 1 là sản phẩm 1902 trong products
5 bản ghi không khớp sản phẩm nào (giá trị bị cấy):


,_src_row,product_id,product_name,category,segment,size,color,price,cogs
0,73,73,HanoiStreet RP-60,Outdoor,Activewear,S,orange,-150,1835.397459
1,194,194,SaigonFlex UC-84,Streetwear,Everyday,XL,yellow,N/A,6509.61927
2,515,515,VietMotion UE-07,Streetwear,Performance,M,blue,999999999,18.38636752042886
3,530,530,MekongFit UE-20,N/A,Performance,S,white,7559.37,5095.771317
4,813,813,SaigonFlex UM-96,Unknown_999,Balanced,XL,black,11060.080544680852,10507.076517446809


product_id rỗng hoặc None:


,_src_row,product_id,product_name,category,segment,size,color,price,cogs
31,32,None,DragonWear UE-02,Streetwear,Performance,L,white,8386.030434782608,7570.06967347826
287,288,,SaigonFlex UM-46,Streetwear,Balanced,M,black,24.20837585679796,14.981117116583182


product_id trùng:


,_src_row,product_id,product_name,category,segment,size,color,price,cogs
403,404,405,BambooCraft UC-25,Streetwear,Everyday,L,green,8808.87,7972.027350000001
404,405,405,VietMode MA-20,Casual,All-weather,S,white,60.9010021903396,38.01870835212474


Giá trị thật của 5 bản ghi bị cấy, tra trong `products` theo tên sản phẩm (dùng cho §6.1 của tài liệu: đoán kiểu "bỏ dấu âm"
sẽ cho 150 thay cho giá thật). Với `tf`, giá trị gốc nằm ở cột `gia_tri_goc` của phần 6.

In [18]:
Q('''
select e._src_row, e.product_name, e.category as category_epd, p.category as category_that,
       e.price as price_epd, p.price as price_that, e.cogs as cogs_epd, p.cogs as cogs_that
from epd e join products p using (product_name)
where e._src_row in (select _src_row from epd_khop where n_khop = 0)
order by e._src_row
''')

,_src_row,product_name,category_epd,category_that,price_epd,price_that,cogs_epd,cogs_that
0,73,HanoiStreet RP-60,Outdoor,Outdoor,-150,2015.37,1835.397459,1835.397459
1,194,SaigonFlex UC-84,Streetwear,Streetwear,N/A,11336.85,6509.61927,6509.61927
2,515,VietMotion UE-07,Streetwear,Streetwear,999999999,29.047579017615437,18.38636752042886,18.38636752042886
3,530,MekongFit UE-20,N/A,Streetwear,7559.37,7559.37,5095.771317,5095.771317
4,813,SaigonFlex UM-96,Unknown_999,Streetwear,11060.080544680852,11060.080544680852,10507.076517446809,10507.076517446809


## 6. `tf.json`: phát lại ngày cũ

So từng ngày với `web_traffic.csv` sau khi bỏ hậu tố ` Variant nnnn` ở `traffic_source`. Số so bằng giá trị số; chuỗi so
nguyên văn, không trim, để thấy cả giá trị bị cấy khoảng trắng hai đầu.

In [19]:
print('cùng tập field với web_traffic:', set(tf.columns) - {'_src_row'} == set(R('web_traffic.csv', nrows=0).columns))
print('traffic_source có hậu tố Variant:', int(tf.traffic_source.fillna('').str.contains(r' Variant \d+$').sum()))
print('date không hợp lệ:', tf[pd.to_datetime(tf.date, format='%Y-%m-%d', errors='coerce').isna()][['_src_row', 'date']].to_dict('records'))
Q('''
select count(*) as ban_ghi,
       count(*) filter (where try_cast(date as date) is null) as date_khong_hop_le,
       count(distinct t.date) filter (where w.date is not null) as ngay_da_co_trong_web_traffic,
       count(*) filter (where try_cast(date as date) is not null and w.date is null) as ngay_moi
from tf t left join web_traffic w using (date)
''')

cùng tập field với web_traffic: True
traffic_source có hậu tố Variant: 997


date không hợp lệ: [{'_src_row': 516, 'date': '2026-99-99'}, {'_src_row': 729, 'date': 'not_a_date'}]


,ban_ghi,date_khong_hop_le,ngay_da_co_trong_web_traffic,ngay_moi
0,1000,2,998,0


In [20]:
NUM = ['sessions', 'unique_visitors', 'page_views', 'bounce_rate', 'avg_session_duration_sec']
parts = [f'''select t._src_row, t.date, '{c}' as cot, t.{c} as gia_tri_tf, w.{c} as gia_tri_goc
             from tf t join web_traffic w using (date)
             where try_cast(t.{c} as double) is null or abs(try_cast(t.{c} as double) - cast(w.{c} as double)) > 1e-9''' for c in NUM]
parts.append('''select t._src_row, t.date, 'traffic_source', t.traffic_source, w.traffic_source
                from tf t join web_traffic w using (date)
                where coalesce(regexp_replace(t.traffic_source, ' Variant \\d+$', ''), '') <> w.traffic_source''')
lech = Q(' union all '.join(parts) + ' order by cot, _src_row')
print('số ô lệch trên các ngày trùng:', len(lech))
lech

số ô lệch trên các ngày trùng: 9


,_src_row,date,cot,gia_tri_tf,gia_tri_goc
0,32,2017-07-29,sessions,NaN,25990
1,108,2013-06-21,sessions,-100,29053
2,220,2016-08-17,sessions,N/A,31419
3,422,2014-09-29,sessions,999999999,18334
4,745,2020-05-23,traffic_source,,email_campaign
5,862,2022-03-01,traffic_source,paid_search Variant 0862,paid_search
6,935,2016-01-15,traffic_source,N/A,paid_search
7,307,2022-11-09,unique_visitors,-50,11080
8,634,2015-11-20,unique_visitors,unknown,10076


## 7. Nếu nạp epd theo id thì COGS ngày lệch bao nhiêu

"Nạp ngây thơ" = ghi đè `products.cogs` bằng `cogs` của epd theo `product_id`, chỉ với dòng epd ép được id và cogs sang số
(id trùng thì lấy dòng sau). Công thức COGS theo CLAUDE.md §5 điểm 1: `COGS(d) = Σ quantity × cogs`, mọi trạng thái đơn.
Dòng đầu là đối chứng với `products` gốc, phải khớp `sales` (lệch chỉ do làm tròn 2 chữ số).
Revenue không đổi vì dùng `order_items.unit_price`, không dùng `products.price`.

In [21]:
con.execute('''
create or replace temp table products_ngay_tho as
with e as (select try_cast(product_id as integer) as pid, try_cast(cogs as double) as cogs, _src_row from epd
           where try_cast(product_id as integer) is not null and try_cast(cogs as double) is not null
           qualify row_number() over (partition by try_cast(product_id as integer) order by _src_row desc) = 1)
select cast(p.product_id as integer) as pid, coalesce(e.cogs, cast(p.cogs as double)) as cogs, e.pid is not null as bi_ghi_de
from products p left join e on cast(p.product_id as integer) = e.pid
''')
def cogs_ngay(products_sql):
    return f'''select cast(o.order_date as date) as d, sum(cast(i.quantity as integer) * pr.cogs) as cogs
               from order_items i join orders o using (order_id) join ({products_sql}) pr on cast(i.product_id as integer) = pr.pid
               group by 1'''
goc = "select cast(product_id as integer) as pid, cast(cogs as double) as cogs from products"
cmp = []
for nhan, sql in [('products gốc (đối chứng)', goc), ('sau khi nạp epd theo id', 'select pid, cogs from products_ngay_tho')]:
    cmp.append(Q(f'''
        with c as ({cogs_ngay(sql)})
        select '{nhan}' as truong_hop, count(*) as so_ngay,
               count(*) filter (where abs(c.cogs - cast(s.COGS as double)) > 1) as ngay_lech_hon_1,
               round(max(abs(c.cogs - cast(s.COGS as double))), 2) as lech_max,
               round(100 * (sum(c.cogs) / sum(cast(s.COGS as double)) - 1), 2) as lech_tong_pct
        from c join sales s on c.d = cast(s.Date as date)'''))
print('sản phẩm bị ghi đè cogs:', one('select count(*) from products_ngay_tho where bi_ghi_de'))
pd.concat(cmp, ignore_index=True)

sản phẩm bị ghi đè cogs: 997


,truong_hop,so_ngay,ngay_lech_hon_1,lech_max,lech_tong_pct
0,products gốc (đối chứng),3833,0,0.0,-0.00
1,sau khi nạp epd theo id,3833,3833,3632537.9,-17.69


## 8. Bằng chứng "trước": hệ thống hiện tại xử lý từng file ra sao

Chạy đúng code trên `main` (commit `8bffc1d`, nhánh `schema/myuyen` chưa sửa code): `scripts/pipeline/snapshot.py`,
`scripts/pipeline/contract.py`, trên **thư mục tạm**. Cell đầu in đường dẫn module để chắc chắn import đúng worktree.
Mỗi ca chỉ đưa vào nguồn liên quan (tham số `contract=` của `prepare`), nên không cần đủ 14 nguồn.

### 8a. `snapshot.prepare`: header, số dòng, tên file

In [22]:
from scripts.ingest.source_contract import SOURCES, COLUMNS, KEYS
from scripts.pipeline import snapshot, contract

print(Path(snapshot.__file__).resolve()); print(Path(contract.__file__).resolve())
TMP = Path(tempfile.mkdtemp(prefix='schema_evo_'))

def as_csv(df, name, dst):
    # Ghi bảng JSON ra CSV theo đúng thứ tự header hợp đồng, None thành ô rỗng (như người dùng tự chuyển tay).
    df[SOURCES[name][0]].to_csv(dst, index=False)

def thu_prepare(ca, files):
    d = TMP / ca / 'data'; d.mkdir(parents=True)
    for dst, src in files.items():
        src(d / dst) if callable(src) else shutil.copy(src, d / dst)
    names = sorted(f[:-4] for f in files if f.endswith('.csv'))
    try:
        out = snapshot.prepare(d, TMP / ca / 'landing', contract={n: SOURCES[n] for n in names})
        m = json.loads((out / 'manifest.json').read_text(encoding='utf-8'))
        return 'ĐẠT', f"snapshot {out.name}; manifest chỉ có {sorted(m['sources'])}"
    except ValueError as e:
        return 'DỪNG', str(e)

cases = {
    '1. orders.csv = nội dung orders_enriched': {'orders.csv': f'{SD}/orders_enriched.csv'},
    '2. shipments.csv = nội dung shipments_realistic': {'shipments.csv': f'{SD}/shipments_realistic.csv'},
    '3. đặt epd.json cạnh products.csv': {'products.csv': f'{DATA}/products.csv', 'epd.json': f'{SD}/epd.json'},
    '4. products.csv = epd chuyển tay sang CSV': {'products.csv': lambda p: as_csv(epd, 'products', p)},
    '5. web_traffic.csv = tf chuyển tay sang CSV': {'web_traffic.csv': lambda p: as_csv(tf, 'web_traffic', p)},
}
ket_qua = pd.DataFrame([(ca, *thu_prepare(f'c{i}', files)) for i, (ca, files) in enumerate(cases.items())],
                       columns=['ca', 'ket_qua', 'thong_bao'])
ket_qua

D:\retail-analytics-schema\scripts\pipeline\snapshot.py
D:\retail-analytics-schema\scripts\pipeline\contract.py


,ca,ket_qua,thong_bao
0,1. orders.csv = nội dung orders_enriched,DỪNG,orders: header sai hợp đồng
1,2. shipments.csv = nội dung shipments_realistic,DỪNG,shipments: header sai hợp đồng
2,3. đặt epd.json cạnh products.csv,ĐẠT,snapshot 90065fbca8fd; manifest chỉ có ['produ...
3,4. products.csv = epd chuyển tay sang CSV,DỪNG,products: số dòng/field sai hoặc nguồn đổi tro...
4,5. web_traffic.csv = tf chuyển tay sang CSV,DỪNG,web_traffic: số dòng/field sai hoặc nguồn đổi ...


Ca 1 dừng vì header có BOM và thừa cột, nhưng thông báo không nói lệch ở đâu. Ca 3 cho thấy file lạ cạnh nguồn
**bị bỏ qua mà không báo gì**. Ca 4–5 dừng ở số dòng cố định, trước khi tới hợp đồng kiểu.

### 8b. `contract.check`: giả sử qua được số dòng, hợp đồng kiểu bắt được gì

Dựng parquet snapshot giống `prepare` (mọi cột là chuỗi, ô rỗng thành null, có `_src_row`) rồi gọi `contract.check` với
hợp đồng hiện có của nguồn tương ứng.

Lưu ý khi đọc kết quả epd: `trùng khóa chính (product_id): 4` gồm 2 dòng id `405` **và** 2 dòng id rỗng, vì phép đếm
`count(*) over (partition by ...)` gom các id null vào cùng một nhóm. Chỉ có một id trùng thật là `405`.

In [23]:
import pyarrow as pa, pyarrow.parquet as pq

def thu_contract(name, df):
    d = TMP / f'contract_{name}'; d.mkdir()
    header = SOURCES[name][0]
    t = pa.table({**{c: pa.array([v if v not in (None, '') else None for v in df[c]], pa.string()) for c in header},
                  '_src_row': pa.array(df._src_row.tolist(), pa.int64())})
    pq.write_table(t, d / f'{name}.parquet')
    summary = contract.check(d, d / 'quarantine', columns={name: COLUMNS[name]}, keys={name: KEYS[name]})
    q = pd.read_csv(d / 'quarantine' / f'{name}.csv', dtype=str) if summary else pd.DataFrame(columns=['_src_row'])
    return summary, q

s_epd, q_epd = thu_contract('products', epd)
print(json.dumps(s_epd, ensure_ascii=False, indent=1))
cay = epd.merge(Q('select _src_row from epd_khop where n_khop = 0'), on='_src_row')
cay.assign(bi_bat=cay._src_row.astype(str).isin(q_epd._src_row))

{
 "products": {
  "rỗng ở cột không được rỗng product_id": 2,
  "không ép được sang double: price": 1,
  "trùng khóa chính (product_id)": 4
 }
}


,_src_row,product_id,product_name,category,segment,size,color,price,cogs,bi_bat
0,73,73,HanoiStreet RP-60,Outdoor,Activewear,S,orange,-150,1835.397459,False
1,194,194,SaigonFlex UC-84,Streetwear,Everyday,XL,yellow,N/A,6509.61927,True
2,515,515,VietMotion UE-07,Streetwear,Performance,M,blue,999999999,18.38636752042886,False
3,530,530,MekongFit UE-20,N/A,Performance,S,white,7559.37,5095.771317,False
4,813,813,SaigonFlex UM-96,Unknown_999,Balanced,XL,black,11060.080544680852,10507.076517446809,False


In [24]:
s_tf, q_tf = thu_contract('web_traffic', tf)
print(json.dumps(s_tf, ensure_ascii=False, indent=1))
lot = lech[~lech._src_row.astype(str).isin(q_tf._src_row)]
print('ô lệch so với bản gốc mà hợp đồng KHÔNG bắt (lọt):')
lot

{
 "web_traffic": {
  "không ép được sang date: date": 2,
  "rỗng ở cột không được rỗng sessions": 1,
  "không ép được sang integer: sessions": 1,
  "không ép được sang integer: unique_visitors": 1,
  "rỗng ở cột không được rỗng traffic_source": 1
 }
}
ô lệch so với bản gốc mà hợp đồng KHÔNG bắt (lọt):


,_src_row,date,cot,gia_tri_tf,gia_tri_goc
1,108,2013-06-21,sessions,-100,29053
3,422,2014-09-29,sessions,999999999,18334
5,862,2022-03-01,traffic_source,paid_search Variant 0862,paid_search
6,935,2016-01-15,traffic_source,N/A,paid_search
7,307,2022-11-09,unique_visitors,-50,11080


In [25]:
print('bản ghi epd khớp đúng 1 sản phẩm nhưng mang id khác, KHÔNG bị hợp đồng bắt:',
      int((~Q("select _src_row from epd_khop where n_khop = 1 and epd_id <> pid_khop")._src_row.astype(str).isin(q_epd._src_row)).sum()))
shutil.rmtree(TMP)

bản ghi epd khớp đúng 1 sản phẩm nhưng mang id khác, KHÔNG bị hợp đồng bắt: 991


## 9. Tóm tắt

Số ở các cell trên thay cho dấu † trong `docs/tien_hoa_schema.md`. Chỗ nào lệch với tài liệu thì **notebook đúng**,
tài liệu phải sửa theo.

Kết luận cho bước "trước":

- Hệ thống hiện tại **chặn** được đổi header (ca 1–2) và đổi số dòng (ca 4–5), nhưng thông báo không chỉ ra lệch ở đâu.
- File mới đặt cạnh nguồn **bị bỏ qua im lặng** (ca 3).
- Hợp đồng kiểu bắt được `'N/A'`, id rỗng/trùng, ngày sai, nhưng **để lọt** giá trị đúng kiểu sai miền và **không thấy**
  việc đánh lại khóa (8b). Với epd, chỉ cổng đối soát G/COGS ở cuối mới chặn được (phần 7); với `web_traffic` thì
  không có cổng nào.